# Dry Bean Dataset - Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

df = pd.read_csv("../Data_sets/Dry_Beans_Dataset.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nStatistical summary:\n{df.describe()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df["Class"].value_counts()
class_counts.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Class Distribution (Count)", fontweight="bold")
axes[0].set_ylabel("Count")

class_counts.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Class Distribution (Percentage)", fontweight="bold")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("../reports/eda_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nClass counts:\n{class_counts}")
print(f"\nClass balance ratio (max/min): {class_counts.max() / class_counts.min():.2f}")

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i], color="steelblue")
    axes[i].set_title(col, fontweight="bold", fontsize=10)
    axes[i].axvline(df[col].mean(), color="red", linestyle="--", label="Mean")
    axes[i].axvline(df[col].median(), color="green", linestyle="--", label="Median")
    skew = df[col].skew()
    axes[i].text(
        0.95, 0.95, f"Skew: {skew:.2f}",
        transform=axes[i].transAxes, ha="right", va="top",
        fontsize=8, bbox=dict(boxstyle="round", facecolor="wheat"),
    )

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/eda_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
corr = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, ax=ax, linewidths=0.5, annot_kws={"size": 7},
)
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/eda_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

high_corr = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.9:
            high_corr.append((corr.columns[i], corr.columns[j], round(corr.iloc[i, j], 3)))

print("\nHighly correlated feature pairs (|r| > 0.9):")
for a, b, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
    print(f"  {a} <-> {b}: r = {r}")

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="lightblue")
    axes[i].set_title(col, fontweight="bold", fontsize=10)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    axes[i].text(
        0.95, 0.95, f"Outliers: {outliers}",
        transform=axes[i].transAxes, ha="right", va="top",
        fontsize=8, bbox=dict(boxstyle="round", facecolor="lightyellow"),
    )

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Outlier Detection (Box Plots)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/eda_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
top_features = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "roundness", "Eccentricity",
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    sns.boxplot(x="Class", y=feat, data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{feat} by Bean Class", fontweight="bold")
    axes[i].tick_params(axis="x", rotation=45)

plt.suptitle("Key Features by Bean Class", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/eda_feature_vs_target.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
top4 = ["Area", "roundness", "Eccentricity", "ShapeFactor2", "Class"]
g = sns.pairplot(df[top4], hue="Class", diag_kind="kde", plot_kws={"alpha": 0.5, "s": 15})
g.fig.suptitle("Pairplot of Top Features", fontsize=14, fontweight="bold", y=1.02)
plt.savefig("../reports/eda_pairplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("=" * 60)
print("EDA KEY FINDINGS")
print("=" * 60)
print(f"""
1. DATASET: {df.shape[0]} samples, {len(numeric_cols)} numeric features, 7 bean classes
2. NO MISSING VALUES - dataset is clean
3. CLASS BALANCE: Reasonably balanced (no extreme imbalance)
4. HIGHLY CORRELATED FEATURES:
   - Area & Perimeter (r > 0.95) -- expected (larger beans = bigger perimeter)
   - MajorAxisLength & Perimeter (r > 0.95)
   - Consider removing redundant features or using PCA
5. SKEWED FEATURES: Area, Perimeter are right-skewed
   - Log transform may help
6. OUTLIERS: Present in Area, Perimeter, and axis measurements
   - RobustScaler or outlier removal may improve results
7. SEPARABILITY: roundness and Eccentricity show good class separation
   - These are likely important features for classification
""")